# Kokoro TTS — Custom Voice Training

Trains a new voice for [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M) by optimising
a `[511, 1, 256]` voice-embedding tensor against your reference audio.

**What you need**
- A **zip file** containing your WAV clips (one clip per utterance, 22–48 kHz, mono or stereo)
- A **`transcript.txt`** file with one line per clip in the format `wavs/1.wav|The text here`
- A free Colab GPU (T4 works fine)

**How it works**
The Kokoro model is fully frozen. Only the voice tensor is optimised, via two differentiable
paths from the mel-spectrogram loss:
- `ref_s[:, :128]` → Decoder AdaIN layers → controls timbre / voice character
- `ref_s[:, 128:]` → ProsodyPredictor (F0, duration, voicing)

**Output** — a `.pt` file you drop straight into any Kokoro pipeline:
```python
for gs, ps, audio in pipeline(text, voice='/path/to/your_voice.pt'):
    ...
```


In [ ]:
# @title Step 1 — Install { run: "auto" }
!pip install -q "kokoro>=0.9.4" soundfile torchaudio
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('Done')

In [ ]:
import os, random, warnings, wave as _wave
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.functional as AF
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display, Audio
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

torch.manual_seed(42)
random.seed(42)

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_SR  = 24000  # Kokoro native rate

## Step 2 — Settings
Change these before running anything else.

In [ ]:
# @title Settings

VOICE_NAME  = 'ff_curie'   # @param {type:"string"}

LANG_CODE   = 'f'  # @param ["a", "b", "e", "f", "h", "i", "p", "j", "z"]
# a=American English  b=British English  e=Spanish  f=French
# h=Hindi  i=Italian  p=Portuguese  j=Japanese  z=Mandarin

BASE_VOICE  = 'ff_siwis'  # @param {type:"string"}

# --- Training ------------------------------------------------------------
NUM_STEPS        = 3000  # @param {type:"integer"}
LEARNING_RATE    = 8e-3  # @param {type:"number"}
GRAD_ACCUM_STEPS = 8     # @param {type:"integer"}
SMOOTHNESS_WEIGHT= 0.005 # @param {type:"number"}
SAVE_EVERY       = 200   # @param {type:"integer"}

# --- Clip duration filter (seconds) -------------------------------------
MIN_DUR = 1.5   # @param {type:"number"}
MAX_DUR = 8.0   # @param {type:"number"}

print(f'Voice name : {VOICE_NAME}')
print(f'Language   : {LANG_CODE}')
print(f'Base voice : {BASE_VOICE or "(none, cold start)"}')
print(f'Steps      : {NUM_STEPS}')


## Step 3 — Mount Google Drive

Mounts your Drive and indexes all WAV files under `/curie-human/wav/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WAV_ROOT = '/content/drive/MyDrive/curie-human/wavs'

# Build filename -> full path index (basename and stem both mapped)
_wav_index = {}
for _root, _dirs, _files in os.walk(WAV_ROOT):
    for _fname in _files:
        if _fname.lower().endswith('.wav'):
            _full = os.path.join(_root, _fname)
            _wav_index[_fname]                       = _full  # '1.wav'
            _wav_index[os.path.splitext(_fname)[0]] = _full  # '1'

print(f'Found {len(_wav_index)//2} WAV files in {WAV_ROOT}')

## Step 4 — Load transcript

Reads `/curie-human/transcript.txt` from Drive.  
Expected format per line: `wavs/1.wav|The text spoken in that clip`  
The directory prefix is ignored — only the filename is used to locate each WAV.

In [ ]:
TRANSCRIPT_PATH = '/content/drive/MyDrive/curie-human/transcript.txt'

with open(TRANSCRIPT_PATH, 'r', encoding='utf-8') as _f:
    _all_lines = [l.strip() for l in _f if l.strip() and '|' in l]

print(f'Loaded {len(_all_lines)} transcript lines from {TRANSCRIPT_PATH}')
print('First few lines:')
for _l in _all_lines[:4]:
    print(f'  {_l[:90]}')

## Step 5 — Load Kokoro model

In [ ]:
from kokoro import KModel, KPipeline

REPO_ID = 'hexgrad/Kokoro-82M'

print('Downloading / loading Kokoro ...')
model = KModel(repo_id=REPO_ID).to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)

# Quiet pipeline for G2P only (no synthesis)
g2p_pipeline = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=False)

print(f'Model loaded ({sum(p.numel() for p in model.parameters()):,} params, all frozen)')

## Step 6 — Build training dataset

Matches each transcript line to its WAV file, resamples to 24 kHz, and phonemises the text.
Clips outside the `MIN_DUR`–`MAX_DUR` window are skipped.

In [ ]:
# ── Diagnostics first ────────────────────────────────────────────────────
print(f'WAV index size   : {len(_wav_index)//2} files')
if _wav_index:
    _sample_keys = list(_wav_index.keys())[:6:2]
    print(f'WAV index sample : {_sample_keys}')
else:
    print('WAV INDEX IS EMPTY -- check WAV_ROOT path in Step 3')

print(f'Transcript lines : {len(_all_lines)}')
if _all_lines:
    _first_rel  = _all_lines[0].split('|')[0].strip()
    _first_base = os.path.basename(_first_rel)
    print(f'First entry path : {_first_rel!r}  ->  basename: {_first_base!r}')
    print(f'Found in index?  : {_wav_index.get(_first_base) or _wav_index.get(os.path.splitext(_first_base)[0])}')
print()

# ── Build training data ───────────────────────────────────────────────────
training_data = []
_reasons = {'not_found': 0, 'duration': 0, 'load': 0, 'phoneme': 0, 'short_ps': 0}

for _line in _all_lines:
    _rel, _text = _line.split('|', 1)
    _text     = _text.strip()
    _basename = os.path.basename(_rel.strip())
    _stem     = os.path.splitext(_basename)[0]
    _wav_path = _wav_index.get(_basename) or _wav_index.get(_stem)

    if _wav_path is None:
        _reasons['not_found'] += 1
        continue

    try:
        with _wave.open(_wav_path) as _w:
            _dur = _w.getnframes() / _w.getframerate()
    except Exception:
        _reasons['duration'] += 1
        continue
    if not (MIN_DUR <= _dur <= MAX_DUR):
        _reasons['duration'] += 1
        continue

    try:
        _wf, _sr = torchaudio.load(_wav_path)
        if _wf.shape[0] > 1:
            _wf = _wf.mean(0, keepdim=True)
        if _sr != TARGET_SR:
            _wf = AF.resample(_wf, _sr, TARGET_SR)
        _pk = _wf.abs().max()
        if _pk > 0:
            _wf = _wf / _pk * 0.95
        _audio = _wf.squeeze(0)
    except Exception:
        _reasons['load'] += 1
        continue

    try:
        _chunks = list(g2p_pipeline(_text, voice=None))
    except Exception:
        _reasons['phoneme'] += 1
        continue
    if not _chunks:
        _reasons['phoneme'] += 1
        continue

    _gs, _ps, _ = _chunks[0]
    _ps = _ps.strip()
    if not _ps or not (2 <= len(_ps) <= 510):
        _reasons['short_ps'] += 1
        continue

    training_data.append({
        'audio':    _audio,
        'phonemes': _ps,
        'ph_len':   len(_ps),
        'text':     _text,
    })

print(f'Built   : {len(training_data)} samples')
print(f'Skipped : {sum(_reasons.values())} total')
for _k, _v in _reasons.items():
    if _v:
        print(f'  {_k:12s}: {_v}')

if not training_data:
    raise RuntimeError('No training samples built. See skip reasons above.')

_ph  = [d['ph_len'] for d in training_data]
_min = sum(len(d['audio']) for d in training_data) / TARGET_SR / 60
print(f'Total audio      : {_min:.1f} min')
print(f'Phoneme lengths  : min={min(_ph)}, max={max(_ph)}, mean={np.mean(_ph):.0f}')
print()
for _it in random.sample(training_data, min(4, len(training_data))):
    print(f'  [{_it["ph_len"]:3d} ph]  "{_it["text"][:70]}"')

## Step 7 — Training utilities

`KModel.forward_with_tokens` is decorated `@torch.no_grad()`. We call submodules directly
so gradients flow through two paths into the voice tensor:

| Slice | Path | Controls |
|-------|------|----------|
| `ref_s[:, :128]` | Decoder AdaIN layers | Timbre / voice character |
| `ref_s[:, 128:]` | ProsodyPredictor (DurationEncoder, F0/N) | Pitch, rhythm, voicing |

In [ ]:
def training_forward(model, phonemes, ref_s, speed=1.0):
    dev = model.device
    ids = [model.vocab[p] for p in phonemes if p in model.vocab]
    if not ids:
        raise ValueError(f'No vocab matches: {phonemes!r}')
    input_ids     = torch.LongTensor([[0, *ids, 0]]).to(dev)
    input_lengths = torch.full((1,), input_ids.shape[-1], dtype=torch.long, device=dev)
    text_mask     = torch.arange(input_lengths.max()).unsqueeze(0).expand(1,-1).type_as(input_lengths)
    text_mask     = torch.gt(text_mask + 1, input_lengths.unsqueeze(1)).to(dev)

    with torch.no_grad():
        bert_dur = model.bert(input_ids, attention_mask=(~text_mask).int())
        d_en     = model.bert_encoder(bert_dur).transpose(-1, -2)
        t_en     = model.text_encoder(input_ids, input_lengths, text_mask)

    s        = ref_s[:, 128:]
    d        = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
    x, _     = model.predictor.lstm(d)
    duration = torch.sigmoid(model.predictor.duration_proj(x)).sum(axis=-1) / speed

    with torch.no_grad():
        pred_dur     = torch.round(duration).clamp(min=1).long().squeeze()
        indices      = torch.repeat_interleave(torch.arange(input_ids.shape[1], device=dev), pred_dur)
        aln          = torch.zeros((input_ids.shape[1], indices.shape[0]), device=dev)
        aln[indices, torch.arange(indices.shape[0])] = 1
        aln          = aln.unsqueeze(0)

    F0, N  = model.predictor.F0Ntrain(d.transpose(-1,-2) @ aln, s)
    audio  = model.decoder(t_en @ aln, F0, N, ref_s[:, :128]).squeeze()
    return audio


_mel_fn = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR, n_fft=1024, hop_length=256,
    win_length=1024, n_mels=80, f_min=0, f_max=8000,
).to(device)

def log_mel(audio):
    return torch.log(_mel_fn(audio.unsqueeze(0).to(device)).squeeze(0).clamp(min=1e-5))

def voice_loss(gen, ref):
    gm, rm = log_mel(gen), log_mel(ref)
    loss   = F.l1_loss(gm.mean(-1), rm.mean(-1))               # global timbre
    loss  += 0.5 * F.l1_loss(gm.std(-1),  rm.std(-1))          # dynamics
    n      = min(gm.shape[-1], rm.shape[-1])
    if n > 8:
        loss += 0.2 * F.l1_loss(gm[:, :n], rm[:, :n])          # frame-level
    return loss

def smooth_reg(pack):
    return torch.mean((pack[1:] - pack[:-1]).pow(2))

print('Utilities ready')

## Step 8 — Initialise voice pack

Shape: `[511, 1, 256]`.  
Warm-starting from an existing voice is strongly recommended — it converges in far fewer steps.

In [ ]:
from torch import nn

if BASE_VOICE:
    print(f'Warm-starting from: {BASE_VOICE}')
    _tmp       = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
    _base_pack = _tmp.load_voice(BASE_VOICE).to(device)
    del _tmp
    voice_pack = nn.Parameter(_base_pack.clone().detach())
else:
    print('Cold start (random)')
    voice_pack = nn.Parameter(torch.randn(511, 1, 256, device=device) * 0.05)

optimizer = torch.optim.Adam([voice_pack], lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_STEPS, eta_min=LEARNING_RATE * 0.01)

print(f'Voice pack: {tuple(voice_pack.shape)}  ({voice_pack.numel():,} parameters)')

## Step 9 — Train

In [ ]:
from tqdm.notebook import tqdm

losses    = []
best_loss = float('inf')
best_pack = voice_pack.detach().clone()

# Mirror checkpoints to Drive so they survive session resets
DRIVE_CKPT_DIR = '/content/drive/MyDrive/curie-human/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

def _save_ckpt(pack, tag):
    for d in [OUTPUT_DIR, DRIVE_CKPT_DIR]:
        path = f'{d}/{VOICE_NAME}_{tag}.pt'
        torch.save(pack.detach().cpu(), path)
    print(f'  [saved] {VOICE_NAME}_{tag}.pt')

for step in tqdm(range(1, NUM_STEPS + 1), desc='Training'):
    optimizer.zero_grad()
    step_loss, valid = 0.0, 0

    for item in random.sample(training_data, min(GRAD_ACCUM_STEPS, len(training_data))):
        ref_s = voice_pack[item['ph_len'] - 1]
        try:
            loss = voice_loss(
                training_forward(model, item['phonemes'], ref_s),
                item['audio'].to(device)
            ) / GRAD_ACCUM_STEPS
            loss.backward()
            step_loss += loss.item()
            valid     += 1
        except Exception as e:
            print(f'  [skip step {step}] {e}')

    # Always checkpoint on schedule, regardless of valid count
    if step % SAVE_EVERY == 0:
        _save_ckpt(voice_pack, f'step{step:04d}')

    if valid == 0:
        continue

    (SMOOTHNESS_WEIGHT * smooth_reg(voice_pack)).backward()
    nn.utils.clip_grad_norm_([voice_pack], 1.0)
    optimizer.step()
    scheduler.step()
    losses.append(step_loss)

    if step_loss < best_loss:
        best_loss = step_loss
        best_pack = voice_pack.detach().clone()

_save_ckpt(best_pack, 'best')
print(f'Done. Best loss: {best_loss:.4f}')


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(losses, lw=0.8, label='loss')
if len(losses) >= 20:
    w  = max(1, len(losses) // 20)
    sm = np.convolve(losses, np.ones(w)/w, mode='valid')
    plt.plot(range(w-1, len(losses)), sm, lw=2, label=f'smoothed')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title(VOICE_NAME); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/{VOICE_NAME}_loss.png', dpi=150)
plt.show()

## Step 10 — Listen and evaluate

In [ ]:
_eval = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)

TEST_SENTENCES = [
    'Hello! This is my new custom voice trained with Kokoro.',
    'The quick brown fox jumps over the lazy dog.',
    'Speech synthesis has come a long way in recent years.',
]  # @param

def synth(pack, text, label=''):
    if label:
        print(f'\n{label}:  "{text[:60]}"')
    for gs, ps, audio in _eval(text, voice=pack.to(device)):
        if audio is not None:
            display(Audio(data=audio.numpy(), rate=TARGET_SR))
            break

for s in TEST_SENTENCES:
    synth(best_pack, s, 'New voice (best)')

if BASE_VOICE:
    print(f'\n--- Base voice ({BASE_VOICE}) for comparison ---')
    for s in TEST_SENTENCES[:1]:
        synth(_eval.load_voice(BASE_VOICE), s)

## Step 11 — Export and download

Saves the best checkpoint as `{VOICE_NAME}.pt` (shape `[511, 1, 256]`) and downloads it.

In [ ]:
_export = best_pack.cpu()
assert _export.shape == (511, 1, 256), f'Bad shape: {_export.shape}'

# Save locally + to Drive
for _d in [OUTPUT_DIR, DRIVE_CKPT_DIR]:
    _out = f'{_d}/{VOICE_NAME}.pt'
    torch.save(_export, _out)
    print(f'Saved: {_out}  ({os.path.getsize(_out)//1024} KB)')

print(f'Shape: {tuple(_export.shape)}')
print(f'Usage:')
print(f'  pipeline(text, voice="{OUTPUT_DIR}/{VOICE_NAME}.pt")')

try:
    from google.colab import files as _cf
    _cf.download(f'{OUTPUT_DIR}/{VOICE_NAME}.pt')
except Exception:
    print(f'(auto-download unavailable — file is at {OUTPUT_DIR}/{VOICE_NAME}.pt and {DRIVE_CKPT_DIR}/{VOICE_NAME}.pt)')


## Step 11b — Export as `.bin` for ONNX

The [Kokoro ONNX runtime](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX) loads voices
as raw `float32` binary files instead of PyTorch `.pt` files.  
The conversion is a single `tofile()` call — no shape or header information is stored;
the loader reconstructs the `[511, 1, 256]` tensor with `np.fromfile(...).reshape(-1, 1, 256)`.

In [ ]:
import numpy as np

_export = best_pack.cpu()

# Save locally + to Drive
for _d in [OUTPUT_DIR, DRIVE_CKPT_DIR]:
    _bin_out = f'{_d}/{VOICE_NAME}.bin'
    _export.numpy().astype(np.float32).tofile(_bin_out)
    print(f'Saved: {_bin_out}  ({os.path.getsize(_bin_out)//1024} KB)')

# Verify
_check = np.fromfile(f'{OUTPUT_DIR}/{VOICE_NAME}.bin', dtype=np.float32).reshape(-1, 1, 256)
assert _check.shape == (511, 1, 256), f'Bad shape: {_check.shape}'
assert np.allclose(_check, _export.numpy(), atol=1e-6)
print(f'Verified shape: {_check.shape}')

try:
    from google.colab import files as _cf
    _cf.download(f'{OUTPUT_DIR}/{VOICE_NAME}.bin')
except Exception:
    print(f'(auto-download unavailable — file is at {OUTPUT_DIR}/{VOICE_NAME}.bin and {DRIVE_CKPT_DIR}/{VOICE_NAME}.bin)')


## Appendix A — Quick test during training
Run at any time without interrupting the training loop.

In [ ]:
# @title Quick test
QUICK_TEXT = 'Hello, this is a quick test.'  # @param {type:"string"}
_p = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
for gs, ps, audio in _p(QUICK_TEXT, voice=voice_pack.detach().to(device)):
    if audio is not None:
        display(Audio(data=audio.numpy(), rate=TARGET_SR)); break

## Appendix B — Resume from a checkpoint
If your Colab session times out, load the last saved checkpoint and re-run Step 9.

In [ ]:
# @title Resume from checkpoint
RESUME_PATH = '/content/output/ff_curie_step0200.pt'  # @param {type:"string"}
if os.path.exists(RESUME_PATH):
    _ld = torch.load(RESUME_PATH, map_location=device, weights_only=True)
    assert _ld.shape == (511, 1, 256)
    voice_pack = nn.Parameter(_ld.detach().clone())
    optimizer  = torch.optim.Adam([voice_pack], lr=LEARNING_RATE)
    print(f'Resumed from {RESUME_PATH}. Re-run Step 9 to continue.')
else:
    print(f'File not found: {RESUME_PATH}')

## Appendix C — Blend voices
Mix your new voice with one or more built-in voices.

In [ ]:
# @title Blend voices
BLEND_WITH  = 'af_heart,af_bella'  # @param {type:"string"}
BLEND_WEIGHT = 0.5                  # @param {type:"slider", min:0, max:1, step:0.05}
# 1.0 = pure new voice, 0.0 = pure built-in

_bp  = KPipeline(lang_code=LANG_CODE, repo_id=REPO_ID, model=model)
_avg = torch.stack([_bp.load_voice(v.strip()) for v in BLEND_WITH.split(',')]).mean(0).to(device)
_blended = (BLEND_WEIGHT * best_pack.to(device) + (1-BLEND_WEIGHT) * _avg).detach()

BLEND_TEXT = 'This is the blended voice.'  # @param {type:"string"}
for gs, ps, audio in _bp(BLEND_TEXT, voice=_blended):
    if audio is not None:
        display(Audio(data=audio.numpy(), rate=TARGET_SR)); break

_bpath = f'{OUTPUT_DIR}/{VOICE_NAME}_blend{int(BLEND_WEIGHT*100)}.pt'
torch.save(_blended.cpu(), _bpath)
print(f'Blend saved: {_bpath}')